# Imports

In [1]:
import pandas as pd
import numpy as np

# Constants

In [2]:
# Raw Data Path
PATH_RAW = "../data/raw/"
# Palpites
FILE_TIPS = "1 - palpites.xlsx"

# Processed Data Path
PATH_PROCESSED = "../data/processed/"

# ETL

In [3]:
# Lendo dados brutos de palpites
df_tips_gs = pd.read_excel(PATH_RAW+FILE_TIPS, sheet_name="palpites_fg",engine="openpyxl")

# Removendo colunas desnecessárias
# Deixando somente o nome e os palpites
df_tips_gs_2 = df_tips_gs.drop(columns=["Carimbo de data/hora","Deixe uma foto sua aqui", "Campeão", "Vice", "Artilheiro"])

In [4]:
# Derrete as colunas em linhas
# col jogo (nome temporario) recebe o confronto
# gols (nome temporario) recebe os valores 
df_tips_gs_pivot = (
    df_tips_gs_2.melt(id_vars=["Nome"], var_name="col_jogo", value_name="gols")
      .dropna(subset=["gols"])
)

In [5]:
# Quebra o texto que está no padrão forms
# Time A x Time B [Time A]
# Vira :
# nm_cfr = Time A x Time B
# nm_time_palpite = Time A
df_tips_gs_pivot["nm_cfr"] = df_tips_gs_pivot["col_jogo"].str.extract(r"^(.*) \[")[0].str.strip()
df_tips_gs_pivot["nm_time_palpite"] = df_tips_gs_pivot["col_jogo"].str.extract(r"\[(.*)\]")[0].str.strip()
# nm_time_casa = Time A
# nm_time_fora = Time B
parts = df_tips_gs_pivot["col_jogo"].str.extract(r"^(.*?) x (.*?) \[")
df_tips_gs_pivot["nm_time_casa"] = parts[0].str.strip()
df_tips_gs_pivot["nm_time_fora"] = parts[1].str.strip()

In [6]:
# Cria um label para definir se o valor vai pra casa ou fora
df_tips_gs_pivot["side"] = df_tips_gs_pivot.apply(
    lambda r: "vl_time_casa" if r["nm_time_palpite"] == str(r["nm_time_casa"]) else "vl_time_fora",
    axis=1,
)
# Unpivota de novo agora para preencher as colunas de vl_fora e vl_casa
df_tips_gs_pivot_unpivot = (
    df_tips_gs_pivot.pivot_table(
        index=["Nome","nm_cfr","nm_time_casa","nm_time_fora"],
        columns="side",
        values="gols",
        aggfunc="first",
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

In [8]:
df_tips_gs_pivot_unpivot["result_ref_casa"] = np.select(
    [
        df_tips_gs_pivot_unpivot["vl_time_casa"] > df_tips_gs_pivot_unpivot["vl_time_fora"],   # vitória do mandante
        df_tips_gs_pivot_unpivot["vl_time_casa"] == df_tips_gs_pivot_unpivot["vl_time_fora"],  # empate
    ],
    ["V", "E"],  # valores para as duas primeiras condições
    default="D"  # derrota do mandante
)

In [11]:
# Ajustando a ordem e renomeando colunas
df_tips_gs_final = df_tips_gs_pivot_unpivot[['Nome','nm_cfr','nm_time_casa','vl_time_casa','nm_time_fora', 'vl_time_fora','result_ref_casa']]
df_tips_gs_final = df_tips_gs_final.rename(columns={'Nome': 'nm_player'})

# Save

In [13]:
# Salvar sem o índice
file_path = PATH_PROCESSED + "palpites__fg_processados.csv"
df_tips_gs_final.to_csv(file_path, index=False, encoding='utf-8')